# Figure 5 Reproduction: JMN1 vs Schwarzschild Accretion Disk Spectrum

This notebook reproduces the spectral luminosity comparison (Figure 5 style) between:
- Schwarzschild spacetime
- JMN1 interior matched to Schwarzschild exterior

It is designed to run both locally and in **Google Colab**.

## Imports

The following packages are required: `numpy`, `scipy`, `sympy`, and `matplotlib`.

In [ ]:
from functools import lru_cache
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import sympy as sp
from scipy.integrate import quad

## Physics Background

For a static, spherically symmetric metric
$$ds^2 = -A(r)dt^2 + B(r)dr^2 + r^2 d\Omega^2,$$
the circular geodesic quantities are
$$E(r)=\sqrt{\frac{2A(r)^2}{2A(r)-rA'(r)}},$$
$$L(r)=\sqrt{\frac{r^3A'(r)}{2A(r)-rA'(r)}},$$
$$\Omega(r)=\sqrt{\frac{A'(r)}{2r}}.$$

The thin-disk flux is computed via
$$F(r) = -\frac{\dot M}{4\pi\sqrt{-g}}\,\frac{\Omega_{,r}}{\left(E-\Omega L\right)^2}
\int_{r_{\rm in}}^{r}\left(E-\Omega L\right)L_{,r}\,dr,$$
with
$$g=-r^2A(r)B(r).$$

The observed redshift factor used in the spectral integrand is
$$z(r)=\frac{1}{\sqrt{-\left(-A(r)+\Omega(r)^2r^2\right)}}-1.$$

## Schwarzschild Metric

For Schwarzschild:
$$A(r)=1-\frac{2M}{r}, \qquad B(r)=\frac{1}{A(r)}.$$

## JMN1 Metric

For JMN1 interior:
$$A(r)=(1-M_0)\left(\frac{r}{R_b}\right)^{\frac{M_0}{1-M_0}}, \qquad B(r)=\frac{1}{1-M_0},$$
with matching radius
$$R_b=\frac{2}{M_0}.$$

## Flux Calculation

This section defines reusable symbolic and numerical utilities for flux computation.
All equations and numerical settings match the script implementation.

In [ ]:
# Global parameters (unchanged)
M = 1.0
MDOT = 1.0
ROUT = 1e4
L_FLOOR = 1e-20

# Schwarzschild-only setup
RIN_SCH_ONLY = 6.0

# JMN1 + Schwarzschild matched setup
M0 = 0.25
RB = 2 / M0
RIN_JMN = 1e-10
RIN_SCH_MATCHED = RB

# Frequency grids (unchanged)
HVKT_SCH = np.logspace(-6, 2, 120)
HVKT_JMN = np.logspace(-6, 2, 80)

# Symbolic radius
R_SYM = sp.symbols('r', positive=True)


def build_metric_functions(a_expr, b_expr):
    apr_expr = sp.diff(a_expr, R_SYM)
    e_expr = sp.sqrt((2 * a_expr**2) / (2 * a_expr - R_SYM * apr_expr))
    l_expr = sp.sqrt((R_SYM**3 * apr_expr) / (2 * a_expr - R_SYM * apr_expr))
    omega_expr = sp.sqrt(apr_expr / (2 * R_SYM))

    domega_expr = sp.diff(omega_expr, R_SYM)
    dl_expr = sp.diff(l_expr, R_SYM)
    gdet_expr = -R_SYM**2 * a_expr * b_expr

    return {
        'A': sp.lambdify(R_SYM, a_expr, 'numpy'),
        'E': sp.lambdify(R_SYM, e_expr, 'numpy'),
        'L': sp.lambdify(R_SYM, l_expr, 'numpy'),
        'Omega': sp.lambdify(R_SYM, omega_expr, 'numpy'),
        'dOmega': sp.lambdify(R_SYM, domega_expr, 'numpy'),
        'dL': sp.lambdify(R_SYM, dl_expr, 'numpy'),
        'gdet': sp.lambdify(R_SYM, gdet_expr, 'numpy'),
    }


def redshift_factor(radius, funcs):
    return 1 / np.sqrt(-(-funcs['A'](radius) + funcs['Omega'](radius)**2 * radius**2)) - 1


def make_flux_function(funcs, rin, mdot, cache_size):
    @lru_cache(maxsize=cache_size)
    def flux(radius):
        integral_val, _ = quad(
            lambda x: (funcs['E'](x) - funcs['Omega'](x) * funcs['L'](x)) * funcs['dL'](x),
            rin,
            radius,
            limit=200,
        )

        prefactor = -(mdot / (4 * np.pi * np.sqrt(-funcs['gdet'](radius))))
        denom = (funcs['E'](radius) - funcs['Omega'](radius) * funcs['L'](radius))**2

        if denom <= 0 or not np.isfinite(denom):
            return 0.0

        return prefactor * (funcs['dOmega'](radius) / denom) * integral_val

    return flux

## Spectral Luminosity

The radial spectral integrand is evaluated and then integrated over radius for each frequency bin.

In [ ]:
def luminosity_integrand(radius, hvkt, funcs, flux_func):
    flux_val = flux_func(radius)
    if flux_val <= 0:
        return 0.0

    z_plus_one = 1 + redshift_factor(radius, funcs)
    expo = np.exp((z_plus_one * hvkt) / flux_val**0.25) - 1
    d_l_inf = 4 * np.pi * radius * np.sqrt(-funcs['gdet'](radius)) * funcs['E'](radius) * flux_val

    return (
        (15 / np.pi**4)
        * d_l_inf
        * ((z_plus_one**4 * hvkt**4) / flux_val)
        / expo
        * (1 / radius)
    )


def compute_schwarzschild_spectrum():
    a_sch_expr = 1 - 2 * M / R_SYM
    b_sch_expr = 1 / a_sch_expr
    sch_funcs = build_metric_functions(a_sch_expr, b_sch_expr)

    flux_sch = make_flux_function(
        funcs=sch_funcs,
        rin=RIN_SCH_ONLY,
        mdot=MDOT,
        cache_size=2000,
    )

    spectrum = []
    for hvkt in HVKT_SCH:
        val, _ = quad(
            lambda rr: luminosity_integrand(rr, hvkt, sch_funcs, flux_sch),
            RIN_SCH_ONLY,
            ROUT,
            limit=100,
        )
        spectrum.append((np.log10(hvkt), np.log10(val)))

    return np.array(spectrum)


def compute_jmn_plus_schwarzschild_spectrum():
    a_jmn_expr = (1 - M0) * (R_SYM / RB) ** (M0 / (1 - M0))
    b_jmn_expr = 1 / (1 - M0)
    jmn_funcs = build_metric_functions(a_jmn_expr, b_jmn_expr)

    a_sch_expr = 1 - 2 * M / R_SYM
    b_sch_expr = 1 / a_sch_expr
    sch_funcs = build_metric_functions(a_sch_expr, b_sch_expr)

    flux_jmn = make_flux_function(
        funcs=jmn_funcs,
        rin=RIN_JMN,
        mdot=MDOT,
        cache_size=50000,
    )
    flux_sch = make_flux_function(
        funcs=sch_funcs,
        rin=RIN_SCH_MATCHED,
        mdot=MDOT,
        cache_size=50000,
    )

    spectrum = []
    for hvkt in HVKT_JMN:
        l_jmn, _ = quad(
            lambda rr: luminosity_integrand(rr, hvkt, jmn_funcs, flux_jmn),
            RIN_JMN,
            RB,
            limit=100,
        )
        l_sch, _ = quad(
            lambda rr: luminosity_integrand(rr, hvkt, sch_funcs, flux_sch),
            RIN_SCH_MATCHED,
            ROUT,
            limit=100,
        )

        l_total = max(l_jmn + l_sch, L_FLOOR)
        spectrum.append((np.log10(hvkt), np.log10(l_total)))

    return np.array(spectrum)


spec_sch = compute_schwarzschild_spectrum()
spec_jmn = compute_jmn_plus_schwarzschild_spectrum()

## Plotting

We generate the comparison plot and save it automatically to `figures/figure5.png`.

In [ ]:
output_dir = Path('figures')
output_dir.mkdir(parents=True, exist_ok=True)
output_path = output_dir / 'figure5.png'

plt.figure()
plt.plot(spec_sch[:, 0], spec_sch[:, 1], '--', label='Schwarzschild')
plt.plot(spec_jmn[:, 0], spec_jmn[:, 1], '--', label='JMN + Schwarzschild')
plt.xlabel('log10(hν / kT*)')
plt.ylabel('log10(ν Lν,∞)')
plt.grid(True, linestyle=':')
plt.xlim(-5, 1)
plt.ylim(-10, 0)
plt.legend()
plt.tight_layout()
plt.savefig(output_path, dpi=300)
plt.show()

print(f'Saved figure to: {output_path.resolve()}')

### Colab Note
If running in Google Colab and you want to download the output file:
```python
from google.colab import files
files.download('figures/figure5.png')
```